# Day 56 — Orchestration basics: Prefect intro
Objectives:
- Create simple Prefect flows and tasks.
- Parameterize a pipeline (preprocess → train → evaluate).
- Run locally and observe logs.
Note: `pip install prefect` (already added to requirements.txt).


In [ ]:
from prefect import flow, task

@task
def load_data():
    import seaborn as sns
    df = sns.load_dataset('titanic').dropna(subset=['survived','sex','class','fare','age'])
    return df

@task
def preprocess(df):
    import pandas as pd
    df = df.copy()
    df['fare'] = pd.to_numeric(df['fare'], errors='coerce').fillna(df['fare'].median())
    df = df.dropna(subset=['age'])
    return df

@task
def train(df):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    from pathlib import Path

    import joblib
    X = df[['sex','class','fare','age']]; y = df['survived']
    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['sex','class']),
                             ('num', StandardScaler(), ['fare','age'])])
    pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])
    Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
    pipe.fit(Xtr,ytr)
    import numpy as np
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(yte, pipe.predict_proba(Xte)[:,1])
    artifact_dir = Path('artifacts/day56')
    artifact_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipe, artifact_dir / 'prefect_titanic_pipeline.joblib')
    return auc

@flow
def training_flow():
    df = load_data()
    df2 = preprocess(df)
    auc = train(df2)
    print('AUC:', auc)

# Run the flow
if __name__ == '__main__':
    training_flow()


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — observable task boundaries, retries, caching, and idempotent orchestration

### Mental model

An orchestrator records workflow state and coordinates retries,
dependencies, schedules, and concurrency. A flow describes the larger
run; tasks are observable units with their own state and policy. The
boundary should be large enough to be meaningful and small enough to
retry without repeating unrelated side effects.

Retries are safe only for transient failures and idempotent operations.
Caching requires a key representing all inputs and code/data meaning,
plus persisted results. Neither retry nor cache makes an unsafe side
effect idempotent automatically.

### Read the API before running it

- **`@flow`:** creates the orchestration boundary and run-level parameters/state.
- **`@task(retries=..., retry_delay_seconds=...)`:** declares task-level retry policy; exception classification still matters.
- **cache key + result persistence:** reuses a completed result only when identity and storage semantics are deliberate.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — derive an idempotent local output identity

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** The key includes every input that changes result meaning, including data and code/version identity where needed.

In [ ]:
import hashlib
import json

inputs = {"data_snapshot": "sales-v3", "model": "ridge", "alpha": 1.0}
canonical = json.dumps(inputs, sort_keys=True, separators=(",", ":"))
key = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]
output_path = f"artifacts/day56/{key}/metrics.json"
print({"cache_key": key, "output_path": output_path})
assert key in output_path

**Expected observation:** Identical canonical inputs produce the same bounded output identity instead of a new timestamped duplicate.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — separate retryable from permanent failures

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** The task's side effects are idempotent or protected by a durable idempotency key.

In [ ]:
class TemporaryUnavailable(RuntimeError):
    pass

class InvalidInput(ValueError):
    pass

def should_retry(error):
    return isinstance(error, (TemporaryUnavailable, TimeoutError))

cases = [TemporaryUnavailable("later"), TimeoutError(), InvalidInput("bad row")]
decisions = [should_retry(error) for error in cases]
print(decisions)
assert decisions == [True, True, False]

**Expected observation:** Only failures that may succeed without changing the input are eligible for retry.

### Debugging and practice ramp

**Common mistake:** Adding retries to every exception or caching a task with undeclared external inputs and side effects.

**Diagnostic:** Inspect task state history, attempt count, exception type, cache key inputs, persisted result, output identity, and cleanup after injected failure.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define observable task boundaries, retries, caching, and idempotent orchestration in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not schedule/deploy a flow until local runs are deterministic, reruns are safe, and ownership/alerts are defined.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Add `test_size` and `random_state` parameters to the training flow.

**Verify:** Practice 1 — observable task boundaries, retries, caching, and idempotent orchestration — run the flow twice with two explicit test_size/random_state pairs, print parameter values, split row counts/hashes, and metrics, and assert a repeated identical pair reproduces the same split.

2. Split the training task into separate train and evaluate tasks with explicit
   outputs.

**Verify:** Practice 2 — observable task boundaries, retries, caching, and idempotent orchestration — make train return a model/artifact identity and evaluate accept that explicit value; print task states and metric, and inject a train failure to prove evaluate does not run on missing output.

3. Explore the optional local Prefect UI and scheduling basics.

**Verify:** Practice 3 — observable task boundaries, retries, caching, and idempotent orchestration — either print an explicit offline-skip result or start the local UI, record the local URL and one completed flow-run ID/state, then stop it cleanly; scheduling remains optional and must not require a cloud account.

### Progressive hints

1. Pass parameters from the flow to the task; log them with the resulting metric
   so a run can be reproduced.
2. Return the fitted pipeline plus held-out arrays or a small typed result.
   Consider whether passing a large dataset between task processes would scale.
3. First prove `training_flow()` succeeds directly. Then run a local server and
   use Prefect 3's current `serve`/deployment workflow—not commands copied from
   older major versions.

The separate solution reinforces retry, notification, and scheduling concepts.
Use the Prefect 3 execution path in this guide and the current learner
environment when translating those concepts.

### Additional mastery practice

Orchestrate explicit, typed tasks whose retries are safe, artifacts are versioned, and failures are observable. A flow wrapper does not repair an unsafe task.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Retry and idempotence:** Add retries to a task that writes an artifact. Make the write idempotent so a failure after writing cannot create duplicate or partially valid outputs.
   **Progressive hint:** Write to a temporary path, validate, then atomically replace a versioned destination. A retry should produce the same logical result.

**Verify:** Retry and idempotence — inject a failure after the first artifact write, then retry; assert exactly one final path/manifest exists, its hash matches a clean run, no partial file remains, and attempt count/state are recorded.

5. **Cache-key design:** Design a task cache key that changes when data fingerprint, code/config, or relevant parameters change, but not when an unrelated log message changes.
   **Progressive hint:** Hash canonical semantic inputs and include a task/schema version. Do not cache a task whose hidden external state is untracked.

**Verify:** Cache-key design — print cache keys for identical inputs, changed data hash, changed code/config, changed relevant parameter, and log-only change; assert equality only for identical/log-only cases and inequality for semantic changes.

6. **Failure observability:** Instrument a three-task flow so logs and a final summary identify run ID, task, safe input version, attempt, elapsed time, artifact ID, and failure category without logging sensitive rows.
   **Progressive hint:** Use structured fields and task/run context. Emit counts and opaque IDs rather than raw feature values.

**Verify:** Failure observability — capture a three-task failure run and assert every event contains run ID, task, safe input version, attempt, elapsed time, artifact ID, and failure category; raw row and secret sentinels must be absent and final state must identify the failed task.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Retry and idempotence


# Practice 5 — Cache-key design


# Practice 6 — Failure observability
